In [3]:
import sys
import os
from pathlib import Path

if 'google.colab' in sys.modules:
    print("Ambiente Colab rilevato. Inizializzazione in corso...")
    
    # 1. Monta Drive (se non già montato)
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
        
    # Imposta la root del Drive su Colab
    BASE_DIR = Path('/content/drive/Shareddrives/Progetti/Progetto_MLDM')

    GIT_BRANCH = 'zone-test'
    
    # 2. Clona/Aggiorna la repo
    REPO_DIR = '/content/crop-spatial-classification'
    if not os.path.exists(REPO_DIR):
        !git clone -b {GIT_BRANCH} https://github.com/SimoRinaldi/crop-spatial-classification.git {REPO_DIR}
    else:
        !cd {REPO_DIR} && git fetch origin && git checkout -B {GIT_BRANCH} origin/{GIT_BRANCH} && git reset --hard origin/{GIT_BRANCH} && git clean -fd
    
    # 3. Aggancia la cartella per permettere gli import
    if REPO_DIR not in sys.path:
        sys.path.append(REPO_DIR)
        
    # 4. Installa i requisiti in modo silenzioso
    %pip install -q -r {REPO_DIR}/requirements.txt
    print("Setup ambiente Colab completato! Branch attivo su Colab: ", GIT_BRANCH)
    
else:
    print("Ambiente Locale rilevato. Procedo con l'esecuzione...")
    
    BASE_DIR = Path.cwd().parent

    if str(BASE_DIR) not in sys.path:
        sys.path.append(str(BASE_DIR))

# =============================
# DEFINIZIONE PATH UNIVERSALI
# =============================
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed'

# Test di verifica per assicurarsi che i dati siano accessibili
if RAW_DIR.exists():
    print(f"✅ Collegamento ai dati riuscito! Cartella raw: {RAW_DIR}")
else:
    print(f"❌ Attenzione: Cartella non trovata in {RAW_DIR}. Verifica il nome, il mount o il path locale.")
     


Ambiente Colab rilevato. Inizializzazione in corso...
branch 'zone-test' set up to track 'origin/zone-test'.
Reset branch 'zone-test'
Your branch is up to date with 'origin/zone-test'.
HEAD is now at 7edddbf test download
Setup ambiente Colab completato! Branch attivo su Colab:  zone-test
✅ Collegamento ai dati riuscito! Cartella raw: /content/drive/Shareddrives/Progetti/Progetto_MLDM/data/raw


# 01 - Estrazione del ground truth e campionamento delle colture
estrae un campione bilanciato di punti per ciascun tipo di coltura e genera la legenda associata.

#### Ricerca dei file TIF per ciascun anno

In [4]:
from pathlib import Path

main_directory = Path(f"{DATA_DIR}/raw/zone_test")
tifs_3035 = {}

for item in sorted(main_directory.iterdir()):
    if item.is_dir():  
        zone_dir = item
        year_dir = sorted(zone_dir.iterdir()).pop()
        if year_dir.is_dir():
            # trova tutti i .tif per questa zona per il primo anno che trova
            tifs_3035[zone_dir.name] = {
                "tif": [str(tif) for tif in year_dir.rglob("*.tif")],
                "xml": [str(xml) for xml in year_dir.rglob("*.xml")]
            }
            print(f"Zona {zone_dir.name}: trovati {len(tifs_3035[zone_dir.name])} file .tif per l'anno {year_dir.name}")
        else:
            print(f"Cartella zona {zone_dir.name} non trovata in {main_directory}")

print(tifs_3035)

Zona albacete_spagna: trovati 2 file .tif per l'anno 2024
Zona jerez_spagna: trovati 2 file .tif per l'anno 2024
Zona rovato: trovati 2 file .tif per l'anno 2024
Zona vicino_capitanata: trovati 2 file .tif per l'anno 2024
{'albacete_spagna': {'tif': ['/content/drive/Shareddrives/Progetti/Progetto_MLDM/data/raw/zone_test/albacete_spagna/2024/CLMS_HRLVLCC_CTY_S2024_R10m_E32N18_03035_V01_R00/CLMS_HRLVLCC_CTY_S2024_R10m_E32N18_03035_V01_R00.tif'], 'xml': ['/content/drive/Shareddrives/Progetti/Progetto_MLDM/data/raw/zone_test/albacete_spagna/2024/CLMS_HRLVLCC_CTY_S2024_R10m_E32N18_03035_V01_R00/CLMS_HRLVLCC_CTY_S2024_R10m_E32N18_03035_V01_R00.tif.aux.xml', '/content/drive/Shareddrives/Progetti/Progetto_MLDM/data/raw/zone_test/albacete_spagna/2024/CLMS_HRLVLCC_CTY_S2024_R10m_E32N18_03035_V01_R00/CLMS_HRLVLCC_CTY_S2024_R10m_E32N18_03035_V01_R00.xml']}, 'jerez_spagna': {'tif': ['/content/drive/Shareddrives/Progetti/Progetto_MLDM/data/raw/zone_test/jerez_spagna/2024/CLMS_HRLVLCC_CTY_S2024_R10m_

#### Riproiezione in coordinate GPS (EPSG:4326)

In [5]:
from pathlib import Path
import rioxarray
from rasterio.enums import Resampling

tifs_4326 = {}

for zone_name, file_paths in tifs_3035.items():
    zone_file_list = []

    for xml_files in file_paths["tif"]:
        # costruisce il percorso del file riproiettato in EPSG:4326
        file_name = xml_files.replace("03035", "4326").replace("raw", "processed")
        file_path = Path(file_name)
        
        # se il file riproiettato esiste già, salta la riproiezione
        if file_path.is_file():
            zone_file_list.append(file_name)
            continue
        
        # crea la cartella di destinazione se non esiste
        file_path.parent.mkdir(parents=True, exist_ok=True)
        
        # apre il file GeoTIFF originale (EPSG:3035)
        raster_3035 = rioxarray.open_rasterio(xml_files)

        # riproiezione in EPSG:4326, usando NEAREST per preservare i codici delle colture
        raster_4326 = raster_3035.rio.reproject("EPSG:4326", resampling=Resampling.nearest)

        # salva il file riproiettato
        raster_4326.rio.to_raster(file_name)
        raster_3035.close()

        zone_file_list.append(file_name)
        
    tifs_4326[zone_name] = zone_file_list

print(f"\nRiproiezione completata per {sum(len(v) for v in tifs_4326.values())} file")


Riproiezione completata per 4 file


#### Creazione e salvataggio legenda delle colture

In [6]:
import pandas as pd
import xml.etree.ElementTree as ET
from src.config import EXCLUDED_CROP_CLASSES

for zone_name, file_paths in tifs_3035.items():
    zone_file_list = []

    for bbox in file_paths["xml"]:
        if ".aux" not in bbox:
            continue

        # prende il primo file trovato
        file_path = bbox
        print(f"Trovato file .aux.xml: {file_path}")

        tree = ET.parse(file_path)
        root = tree.getroot()

        summary = {}

        # estrae il codice e il nome della coltura
        for r in root.findall(".//Row"):
            fields = r.findall("F")
            
            crop_code = int(fields[0].text)     

            if crop_code in EXCLUDED_CROP_CLASSES:
                continue
            
            crop_count = fields[1].text
            crop_name = fields[2].text

            summary[crop_code] = {"name": crop_name, "count": crop_count}

        df_summary = (
            pd.DataFrame.from_dict(summary, orient="index")
            .rename_axis("crop_code")
            .reset_index()
        )

        print(f"Zona: {item.name}")
        display(df_summary)

Trovato file .aux.xml: /content/drive/Shareddrives/Progetti/Progetto_MLDM/data/raw/zone_test/albacete_spagna/2024/CLMS_HRLVLCC_CTY_S2024_R10m_E32N18_03035_V01_R00/CLMS_HRLVLCC_CTY_S2024_R10m_E32N18_03035_V01_R00.tif.aux.xml
Zona: vicino_capitanata


,crop_code,name,count
0,1110,Wheat,3556780
1,1120,Barley,10572503
2,1130,Maize,866668
3,1140,Rice,0
4,1150,Other Cereals,7383710
5,1210,Fresh Vegetables,2017232
6,1220,Dry Pulses,2750707
7,1310,Potatoes,90132
8,1320,Sugar Beet,25
9,1410,Sunflower,198735


Trovato file .aux.xml: /content/drive/Shareddrives/Progetti/Progetto_MLDM/data/raw/zone_test/jerez_spagna/2024/CLMS_HRLVLCC_CTY_S2024_R10m_E28N16_03035_V01_R00/CLMS_HRLVLCC_CTY_S2024_R10m_E28N16_03035_V01_R00.tif.aux.xml
Zona: vicino_capitanata


,crop_code,name,count
0,1110,Wheat,3614311
1,1120,Barley,567641
2,1130,Maize,260408
3,1140,Rice,87895
4,1150,Other Cereals,1894213
5,1210,Fresh Vegetables,1611515
6,1220,Dry Pulses,1075100
7,1310,Potatoes,128206
8,1320,Sugar Beet,0
9,1410,Sunflower,2833350


Trovato file .aux.xml: /content/drive/Shareddrives/Progetti/Progetto_MLDM/data/raw/zone_test/rovato/2024/CLMS_HRLVLCC_CTY_S2024_R10m_E43N24_03035_V01_R00/CLMS_HRLVLCC_CTY_S2024_R10m_E43N24_03035_V01_R00.tif.aux.xml
Zona: vicino_capitanata


,crop_code,name,count
0,1110,Wheat,8670783
1,1120,Barley,3002656
2,1130,Maize,23153475
3,1140,Rice,328171
4,1150,Other Cereals,225989
5,1210,Fresh Vegetables,3158189
6,1220,Dry Pulses,283663
7,1310,Potatoes,72002
8,1320,Sugar Beet,47437
9,1410,Sunflower,332955


Trovato file .aux.xml: /content/drive/Shareddrives/Progetti/Progetto_MLDM/data/raw/zone_test/vicino_capitanata/2024/CLMS_HRLVLCC_CTY_S2024_R10m_E47N21_03035_V01_R00/CLMS_HRLVLCC_CTY_S2024_R10m_E47N21_03035_V01_R00.tif.aux.xml
Zona: vicino_capitanata


,crop_code,name,count
0,1110,Wheat,1602002
1,1120,Barley,670789
2,1130,Maize,12332
3,1140,Rice,188
4,1150,Other Cereals,39137
5,1210,Fresh Vegetables,494150
6,1220,Dry Pulses,531316
7,1310,Potatoes,10912
8,1320,Sugar Beet,0
9,1410,Sunflower,340336


In [7]:
import xml.etree.ElementTree as ET

for zone_name, file_paths in tifs_3035.items():
    if not isinstance(file_paths, dict) or "xml" not in file_paths:
        continue

    for bbox in file_paths["xml"]:
        if ".aux" in str(bbox):
            continue

        tree = ET.parse(bbox)
        root = tree.getroot()

        bbox_node = root.find(".//{*}EX_GeographicBoundingBox")

        if bbox_node is not None:
            bbox = [
                float(bbox_node.find("{*}westBoundLongitude/{*}Decimal").text),
                float(bbox_node.find("{*}southBoundLatitude/{*}Decimal").text),
                float(bbox_node.find("{*}eastBoundLongitude/{*}Decimal").text),
                float(bbox_node.find("{*}northBoundLatitude/{*}Decimal").text),
            ]
            print(f"Zona: {zone_name} -> BBox: {bbox}")
            file_paths["bbox"] = bbox

print(tifs_3035)


Zona: albacete_spagna -> BBox: [-2.9952, 38.4339, -1.6912, 39.4732]
Zona: jerez_spagna -> BBox: [-7.065, 35.955, -5.7666, 37.0341]
Zona: rovato -> BBox: [9.7312, 44.7065, 11.0113, 45.6125]
Zona: vicino_capitanata -> BBox: [14.5536, 41.8387, 15.8382, 42.8017]
{'albacete_spagna': {'tif': ['/content/drive/Shareddrives/Progetti/Progetto_MLDM/data/raw/zone_test/albacete_spagna/2024/CLMS_HRLVLCC_CTY_S2024_R10m_E32N18_03035_V01_R00/CLMS_HRLVLCC_CTY_S2024_R10m_E32N18_03035_V01_R00.tif'], 'xml': ['/content/drive/Shareddrives/Progetti/Progetto_MLDM/data/raw/zone_test/albacete_spagna/2024/CLMS_HRLVLCC_CTY_S2024_R10m_E32N18_03035_V01_R00/CLMS_HRLVLCC_CTY_S2024_R10m_E32N18_03035_V01_R00.tif.aux.xml', '/content/drive/Shareddrives/Progetti/Progetto_MLDM/data/raw/zone_test/albacete_spagna/2024/CLMS_HRLVLCC_CTY_S2024_R10m_E32N18_03035_V01_R00/CLMS_HRLVLCC_CTY_S2024_R10m_E32N18_03035_V01_R00.xml'], 'bbox': [-2.9952, 38.4339, -1.6912, 39.4732]}, 'jerez_spagna': {'tif': ['/content/drive/Shareddrives/Proge

In [8]:
import rasterio
from src.config import CAPITANATA_BBOX, VICINO_CAPITANATA_BBOX, YEARS_TO_FETCH
from src.sentinel2_downloader import download_area

for zone_name, file_paths in tifs_3035.items():
    if not isinstance(file_paths, dict) or "bbox" not in file_paths:
        continue

    bbox = file_paths["bbox"]

    out_directory = DATA_DIR / "processed" / "zone_test" / f"sentinel2_{zone_name}_area"

    # avvio del download
    downloaded_files = download_area(
        name=zone_name,
        bbox=bbox,
        years=["2024"],
        months=range(1, 13),
        out_dir=str(out_directory)
    )

    # riepilogo dei dati scaricati
    tif_files = sorted(list(out_directory.glob("*.tif")))

    print("\n" + "-" * 50)
    print(f"Riepilogo dati satellitari zona {zone_name}")
    print("-" * 50)
    print(f"Cartella: {out_directory.resolve()}")
    print(f"File GeoTIFF mensili trovati: {len(tif_files)} (attesi: {len(YEARS_TO_FETCH) * 12})")

    total_size_mb = 0.0
    for f in tif_files:
        size_mb = f.stat().st_size / (1024 * 1024)
        total_size_mb += size_mb
        print(f"  • {f.name} ({size_mb:.1f} MB)")

    print(f"Spazio totale occupato: {total_size_mb / 1024:.2f} GB")

    # controllo di integrità sul primo file disponibile
    if tif_files:                                                
        with rasterio.open(tif_files[0]) as src:                 
            print(f"\nVerifica formato raster ({tif_files[0].name}):")                                                      
            print(f"  • Dimensioni: {src.width} x {src.height} pixel")                                                        
            print(f"  • Numero bande: {src.count}") 

Inizio download area 'albacete_spagna' per l'anno 2024

Scaricando albacete_spagna per 2024-01...
  • Scene selezionate per coprire la Capitanata: 6 tile (['MGRS-30SWJ', 'MGRS-30SXJ', 'MGRS-30SXH', 'MGRS-30SWH', 'MGRS-30SVJ', 'MGRS-30SVH'])
    -> [1/6] Elaborazione banda 'blue'...


KeyboardInterrupt: 